<p>
  <img src="../assets/latincy-ext-logo.jpg" alt="LatinCy Ext" width="380">
</p>

# LiLa Lemma Bank Linker — demo

An **offline** component that resolves any lemma emitted by the LatinCy pipeline to a
[LiLa Lemma Bank](https://lila-erc.eu/) URI — no runtime network calls, just a local SQLite artifact.

This notebook walks through the v1 component end to end:

1. **Live linking** — a Latin sentence → LiLa URIs via the spaCy `lila_linker` pipe
2. **Orthographic robustness** — classical `v`/`j` spelling resolves to the same URI as `u`/`i`
3. **Inside the knowledge base** — what the artifact actually contains
4. **Coverage spike (Mode A)** — model-predicted lemma+POS on held-out text → the **~99.4%** headline
5. **Oracle eval (Mode B)** — gold lemma+POS → MFS top-1, the table-quality ceiling
6. **Ambiguity & the v2 verdict** — why the expensive disambiguator isn't worth building

---

> **Licensing.** Backbone = LiLa Lemma Bank (CC-BY-SA 4.0). The enriched artifact carries
> attested links from LASLA (CC-BY-NC-SA 4.0), so the combined artifact is **CC-BY-NC-SA 4.0**.
> Academic / non-commercial use makes the NC clause immaterial.

## Setup

The component is now packaged as `latincy_ext` — install with `pip install -e path/to/latincy-ext`
(editable) or `pip install latincy-ext` once on PyPI. The SQLite artifact and
coverage-spike analysis scripts still live in the `latincy-treebanks` worktree
(they are not bundled with the package). Run this notebook in place from `latincy-ext/notebooks/`.

In [1]:
import os, sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# SQLite artifact and analysis helpers live in the latincy-treebanks worktree.
# Override with LILA_TREEBANKS if your worktree is elsewhere.
TB = Path(os.environ.get(
    "LILA_TREEBANKS",
    "../../latincy-treebanks/.worktrees/lila-lemmabank-linker",
)).resolve()
assert TB.exists(), (
    f"treebanks worktree not found: {TB}\n"
    "  create it: cd latincy-treebanks && git worktree add "
    ".worktrees/lila-lemmabank-linker lila-lemmabank-linker"
)

# analysis helpers (coverage spike / oracle eval) still live in the worktree
LILA_DIR = TB / "scripts" / "lila"
sys.path.insert(0, str(LILA_DIR))

# prefer the full-corpus artifact; fall back to the smaller one
DB = TB / "data" / "lila_linkbank_full.sqlite"
if not DB.exists():
    DB = TB / "data" / "lila_linkbank.sqlite"
DB = str(DB)
HELDOUT = str(TB / "data" / "lila_cache" / "conllu" / "Catullus.conllup")

assert os.path.exists(DB), f"artifact not found under {TB/'data'} — build it with scripts/lila/build_full.sh"
print("artifact :", DB, f"({os.path.getsize(DB)/1e6:.0f} MB)")
print("held-out :", HELDOUT, "(exists)" if os.path.exists(HELDOUT) else "(MISSING)")

URI_PREFIX = "http://lila-erc.eu/data/id/"
def short(uri):
    return uri.replace(URI_PREFIX, "…/") if uri else "—"

artifact : …/data/lila_linkbank_full.sqlite (120 MB)
held-out : …/data/lila_cache/conllu/Catullus.conllup (exists)


## 1 · Live linking

Load `la_core_web_lg`, attach the `lila_linker` pipe, and run Caesar's opening line.
Every content token gets a `Token._.lila_uri`; ambiguous tokens also expose a ranked
`Token._.lila_candidates` list (the v2 hook). `lila_source` records *how* it resolved
(`lemma_pos` → POS-keyed match, the strongest signal).

In [2]:
import spacy
import latincy_ext  # registers the lila_linker @Language.factory

nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("lila_linker", config={"db_path": DB})

doc = nlp("Gallia est omnis divisa in partes tres.")

print(f"{'TOKEN':<10}{'LEMMA':<10}{'POS':<7}{'SOURCE':<11}URI")
print("-" * 66)
for t in doc:
    nc = len(t._.lila_candidates or [])
    amb = f"  (+{nc-1} cand)" if nc > 1 else ""
    print(f"{t.text:<10}{t.lemma_:<10}{t.pos_:<7}{(t._.lila_source or '—'):<11}{short(t._.lila_uri)}{amb}")

TOKEN     LEMMA     POS    SOURCE     URI
------------------------------------------------------------------
Gallia    Gallia    PROPN  lemma_pos  …/lemma/7760
est       sum       AUX    lemma_pos  …/lemma/126689  (+2 cand)
omnis     omnis     DET    lemma_pos  …/lemma/114954
diuisa    diuido    VERB   lemma_pos  …/lemma/99932
in        in        ADP    lemma_pos  …/lemma/106748
partes    pars      NOUN   lemma_pos  …/lemma/116084
tres      tres      NUM    lemma_pos  …/lemma/128764  (+1 cand)
.         .         PUNCT  —          —


## 2 · Orthographic robustness

LiLa keys are normalized (lowercase, `v→u`, `j→i`, diacritics stripped). The same
`normalize_lemma()` runs at build time and at lookup time, with a byte-parity test,
so classical spelling resolves to the **same** URI as the LiLa orthography. Here we hit
the resolver directly (no pipeline) to isolate the lookup.

In [3]:
from latincy_ext.lila_linker import LilaResolver, normalize_lemma

R = LilaResolver(DB)

pairs = [("divido", "VERB"), ("diuido", "VERB"),   # v vs u
         ("Juppiter", "PROPN"), ("Iuppiter", "PROPN"),  # j vs i
         ("VIRTVS", "NOUN"), ("virtus", "NOUN")]      # case + v

print(f"{'INPUT':<12}{'NORMALIZED':<12}{'POS':<7}URI")
print("-" * 52)
for lemma, pos in pairs:
    res = R.resolve(lemma, pos)
    print(f"{lemma:<12}{normalize_lemma(lemma):<12}{pos:<7}{short(res.uri)}")

INPUT       NORMALIZED  POS    URI
----------------------------------------------------
divido      diuido      VERB   …/lemma/99932
diuido      diuido      VERB   …/lemma/99932
Juppiter    iuppiter    PROPN  …/lemma/10964
Iuppiter    iuppiter    PROPN  …/lemma/10964
VIRTVS      uirtus      NOUN   …/lemma/130356
virtus      uirtus      NOUN   …/lemma/130356


## 3 · Inside the knowledge base

The artifact is a plain SQLite file. The backbone (complete lemma → URI map +
orthographic variants) comes from the LiLa MariaDB dump; it's *enriched* with
attested `(lemma, UPOS) → URI` links harvested from CoNLL-U corpora (`is_gold=1`).

In [4]:
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

print("meta:")
for k, v in con.execute("SELECT k, v FROM meta").fetchall():
    print(f"  {k:20} {v}")

print("\ntable sizes:")
for tbl in ("lemma_uri", "wr_uri", "form_uri"):
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:10} {n:>9,} rows")

gold = con.execute("SELECT COUNT(*) FROM lemma_uri WHERE is_gold=1").fetchone()[0]
print(f"\n  of lemma_uri, {gold:,} are gold (corpus-attested)")
con.close()

meta:
  backbone_source      LiLa_Lemma-Bank lila_db.sql
  backbone_license     CC-BY-SA 4.0
  lemmas               167994
  ipolemmas            95685

table sizes:
  lemma_uri    268,713 rows
  wr_uri       336,020 rows
  form_uri     141,425 rows

  of lemma_uri, 28,096 are gold (corpus-attested)


## 4 · Coverage spike — Mode A (end-to-end, model-predicted)

The go/no-go gate. Run held-out **raw text** (Catullus, never seen in the backbone build)
through `la_core_web_lg`, then resolve each *model-predicted* lemma+POS. This measures the
real story: does our lemmatizer's output land on LiLa keys? Headline result: **~99% coverage**.

In [5]:
from collections import Counter
from coverage_spike import _read_text_lines

texts = _read_text_lines(HELDOUT)
src = Counter()
n = hits = ambig = 0
for d in nlp.pipe(texts, batch_size=64):
    for t in d:
        if t.is_punct or t.is_space:
            continue
        n += 1
        res = R.resolve(t.lemma_, t.pos_, t.text)
        src[res.source] += 1
        if res.uri:
            hits += 1
            if len(res.candidates) > 1:
                ambig += 1

print(f"held-out sentences        : {len(texts)}")
print(f"tokens (non-punct)        : {n:,}")
print(f"resolved to a URI         : {hits:,} ({100*hits/n:.1f}%)   ← coverage")
print(f"  of those, ambiguous     : {ambig:,} ({100*ambig/max(hits,1):.1f}%)")
print("source breakdown          :")
for s, c in src.most_common():
    print(f"    {s:10} {c:>6,} ({100*c/n:.1f}%)")

held-out sentences        : 727
tokens (non-punct)        : 13,065
resolved to a URI         : 12,987 (99.4%)   ← coverage
  of those, ambiguous     : 2,842 (21.9%)
source breakdown          :
    lemma_pos  12,539 (96.0%)
    lemma         313 (2.4%)
    wr            135 (1.0%)
    form           47 (0.4%)
    miss           31 (0.2%)


## 5 · Oracle eval — Mode B (table quality)

Now feed the held-out file's **gold** lemma+POS straight from its CoNLL-U-Plus columns and
compare the resolver's top pick against the gold URI. This isolates *table* quality from any
lemmatizer/tokenizer noise — the top-1 number is the **Most-Frequent-Sense (MFS) baseline**
that a v2 disambiguator would have to beat.

In [6]:
from extract_conllu import _token_uri

n = cov = top1 = 0
with open(HELDOUT, encoding="utf-8", errors="replace") as fh:
    for line in fh:
        if not line.strip() or line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")
        if len(cols) < 4 or "-" in cols[0] or "." in cols[0]:
            continue
        lemma, upos = cols[2], cols[3]
        gold_uri, _ = _token_uri(cols)
        if not gold_uri:
            continue
        n += 1
        res = R.resolve(lemma, upos)
        if res.uri:
            cov += 1
            if res.uri == gold_uri:
                top1 += 1

print(f"gold-linked tokens        : {n:,}")
print(f"coverage (URI returned)   : {cov:,} ({100*cov/n:.1f}%)")
print(f"top-1 == gold URI         : {top1:,} ({100*top1/n:.1f}% of all; {100*top1/max(cov,1):.1f}% of covered)")
print("\n↑ this top-1 number is the MFS baseline v2 must beat.")

gold-linked tokens        : 13,129
coverage (URI returned)   : 13,129 (100.0%)
top-1 == gold URI         : 12,996 (99.0% of all; 99.0% of covered)

↑ this top-1 number is the MFS baseline v2 must beat.


## 6 · Ambiguity & the v2 verdict

How much work is left for a sense disambiguator? Group the gold links by `(lemma, UPOS)`,
count keys mapping to more than one URI, and weight by attestation frequency. The slice that
*could* be improved is small and **concentrated** — and most of it is morphosyntactic
(`ut`+indicative vs `ut`+subjunctive, decidable from the governed verb's mood already in FEATS),
not lexical. Hence the verdict: **ship v1; a cheap FEATS/dep rule on ~20–50 keys beats a
biencoder at ~zero cost.**

In [7]:
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
rows = con.execute(
    """SELECT norm_key, upos, COUNT(DISTINCT uri) AS ncand, SUM(freq) AS tok
       FROM lemma_uri WHERE is_gold=1 AND upos<>''
       GROUP BY norm_key, upos"""
).fetchall()

keys = len(rows)
amb_keys = sum(1 for r in rows if r[2] > 1)
tok_total = sum(r[3] or 0 for r in rows)
tok_amb = sum((r[3] or 0) for r in rows if r[2] > 1)

print(f"gold (lemma,UPOS) keys    : {keys:,}")
print(f"  ambiguous (>1 URI)      : {amb_keys:,} ({100*amb_keys/keys:.1f}% of keys)")
print(f"gold attested tokens      : {tok_total:,}")
print(f"  on an ambiguous key     : {tok_amb:,} ({100*tok_amb/tok_total:.1f}% of tokens)  ← v2 workload")
print(f"  v1 already unique       : {100*(tok_total-tok_amb)/tok_total:.1f}% of tokens")

print("\ntop ambiguous keys by attestation (where the addressable error concentrates):")
top = sorted([r for r in rows if r[2] > 1], key=lambda r: -(r[3] or 0))[:12]
for nk, upos, nc, tok in top:
    uris = con.execute(
        "SELECT uri, freq FROM lemma_uri WHERE norm_key=? AND upos=? AND is_gold=1 ORDER BY freq DESC",
        (nk, upos),
    ).fetchall()
    ids = ", ".join(f"{u.rsplit('/',1)[1]}({f})" for u, f in uris[:4])
    print(f"  {nk:12} {upos:5} {nc} URIs, {tok:>6} tok: {ids}")
con.close()

gold (lemma,UPOS) keys    : 27,630
  ambiguous (>1 URI)      : 383 (1.4% of keys)
gold attested tokens      : 2,076,288
  on an ambiguous key     : 405,917 (19.6% of tokens)  ← v2 workload
  v1 already unique       : 80.4% of tokens

top ambiguous keys by attestation (where the addressable error concentrates):
  sum          AUX   3 URIs,  55140 tok: 126689(55131), 126965(8), 123840(1)
  qui          PRON  2 URIs,  53806 tok: 121354(53780), 121318(26)
  que          CCONJ 2 URIs,  31691 tok: 131416(31690), 101354(1)
  ille         DET   2 URIs,  17881 tok: 106221(17879), 139991(2)
  ut           SCONJ 4 URIs,  17833 tok: 130906(13927), 130905(3902), 131110(3), 131025(1)
  quis         PRON  2 URIs,  15345 tok: 121165(15339), 121334(6)
  si           SCONJ 4 URIs,  13878 tok: 124676(13396), 102562(340), 102560(102), 138600(40)
  cum          SCONJ 2 URIs,  10673 tok: 97202(10672), 140014(1)
  res          NOUN  2 URIs,  10613 tok: 121868(10607), 122962(6)
  dico         VERB  5 URIs,  1

## Takeaways

- **It works, offline, today.** Any LatinCy lemma resolves to a LiLa URI from a single local SQLite file — no network, no API risk.
- **Coverage is essentially solved.** ~99% end-to-end on held-out text; classical orthography handled by shared normalization.
- **MFS is already excellent**, so the v2 disambiguation ceiling is tiny (~0.76% of all tokens on the full corpus) and concentrated in a handful of mostly *morphosyntactic* keys.
- **Recommendation:** ship v1; if v2 is ever pursued, a small FEATS/dependency rule over the concentrated tail beats a trained biencoder at near-zero cost.